# Evaluation
## Prepare the environment

In [1]:
LOCAL = True

In [2]:
import torch

import numpy as np

from pathlib import Path

from dataset import MotionDataset
from sample import denormalize_samples, generate
from metrics import compute_fmd, compute_mpjpe, compute_mpjpe_v2, compute_sample_variance

/home/dominika/Desktop/26L_sem8_mgr1/SIGK/AIComputerGraphics/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if LOCAL:
    DATA_PATH = Path("data/train.npz")
    NORM_STATS_PATH = Path("data/norm_stats.npy")
    BEST_MODEL_PATH = Path("best_model/best_model.pt")
else:
    raise NotImplementedError("This code is meant to be run locally.")

print(f"Data: {DATA_PATH} (exists: {DATA_PATH.exists()})")
print(f"Norm stats: {NORM_STATS_PATH} (exists: {NORM_STATS_PATH.exists()})")
print(f"Model: {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.exists()})")

Data: data/train.npz (exists: True)
Norm stats: data/norm_stats.npy (exists: True)
Model: best_model/best_model.pt (exists: True)


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Load the data

In [5]:
dataset = MotionDataset(str(DATA_PATH))
seq, label = dataset[0]

print(f"Samples: {len(dataset)}")
print(f"Sequence shape: {seq.shape}")
print(f"Num classes: {int(dataset.labels.max()) + 1}")

sequences = dataset.sequences.numpy().astype(np.float32)
labels = dataset.labels.numpy().astype(np.int64)
n_frames = sequences.shape[1]
n_joints = sequences.shape[2]
print(f"Test set: {sequences.shape} labels: {np.bincount(labels)}")

Samples: 1260
Sequence shape: torch.Size([48, 15, 3])
Num classes: 2
Test set: (1260, 48, 15, 3) labels: [623 637]


In [6]:
seq_tensor = denormalize_samples(torch.tensor(sequences), str(NORM_STATS_PATH))
sequences = seq_tensor.numpy()
stats = np.load(NORM_STATS_PATH)
print(f"Denormalised real data  (mean={stats[0]:.4f}, std={stats[1]:.4f})")

Denormalised real data  (mean=-0.6862, std=10.6511)


In [7]:
real_walk_mask = labels == 0
real_walk = sequences[real_walk_mask]
print(f"Real walk samples : {len(real_walk)}")

real_jump_mask = labels == 1
real_jump = sequences[real_jump_mask]
print(f"Real jump samples : {len(real_jump)}")

Real walk samples : 623
Real jump samples : 637


## Generate predictions

In [8]:
walk_generated = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=0,
    n_samples=len(real_walk),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

walk_gen = denormalize_samples(walk_generated, str(NORM_STATS_PATH)).numpy()
print(f"Generated walk samples (denormalised) (shape: {walk_gen.shape})")

Generated walk samples (denormalised) (shape: (623, 48, 15, 3))


In [9]:
jump_generated = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=1,
    n_samples=len(real_jump),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

jump_gen = denormalize_samples(jump_generated, str(NORM_STATS_PATH)).numpy()
print(f"Generated jump samples (denormalised) (shape: {jump_gen.shape})")

Generated jump samples (denormalised) (shape: (637, 48, 15, 3))


## Calculate metrics

In [10]:
walk_fmd = compute_fmd(real_walk, walk_gen)
jump_fmd = compute_fmd(real_jump, jump_gen)

print(f"FMD (Fréchet Motion Distance)")
print("  ↳ Lower = generated distribution closer to real.")
print(f"  Walk: {walk_fmd:.4f}")
print(f"  Jump: {jump_fmd:.4f}")

FMD (Fréchet Motion Distance)
  ↳ Lower = generated distribution closer to real.
  Walk: 35.0393
  Jump: 149.2054


In [11]:
mpjpe_nn_walk = compute_mpjpe(real_walk, walk_gen)
mpjpe_mean_walk = compute_mpjpe_v2(real_walk, walk_gen)

mpjpe_nn_jump = compute_mpjpe(real_jump, jump_gen)
mpjpe_mean_jump = compute_mpjpe_v2(real_jump, jump_gen)

print(f"MPJPE (nearest-neighbour pairing)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_nn_walk:.4f}")
print(f"  Jump: {mpjpe_nn_jump:.4f}")
print()
print(f"MPJPE (vs mean real pose)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_mean_walk:.4f}")
print(f"  Jump: {mpjpe_mean_jump:.4f}")

MPJPE (nearest-neighbour pairing)
  ↳ Lower = generated joints closer to real joints.
  Walk: 2.7084
  Jump: 2.1211

MPJPE (vs mean real pose)
  ↳ Lower = generated joints closer to real joints.
  Walk: 19.8097
  Jump: 6.1609


In [12]:
var_walk = compute_sample_variance(walk_gen)
var_jump = compute_sample_variance(jump_gen)

print(f"Sample Diversity:")
print("  ↳ Higher = more creative / diverse outputs.")
print()
print(f"  Mean pairwise distance (feat)")
print(f"  Walk: {var_walk['mean_pairwise_dist']:.4f}")
print(f"  Jump: {var_jump['mean_pairwise_dist']:.4f}")
print()
print(f"  Joint position std (across samp)")
print(f"  Walk: {var_walk['joint_position_std']:.4f}")
print(f"  Jump: {var_jump['joint_position_std']:.4f}")
print()
print(f"  Velocity std (across samp)")
print(f"  Walk: {var_walk['velocity_std']:.4f}")
print(f"  Jump: {var_jump['velocity_std']:.4f}")

Sample Diversity:
  ↳ Higher = more creative / diverse outputs.

  Mean pairwise distance (feat)
  Walk: 52.0644
  Jump: 22.5965

  Joint position std (across samp)
  Walk: 9.5103
  Jump: 3.3395

  Velocity std (across samp)
  Walk: 0.9654
  Jump: 0.4784


## Summary

In [13]:
results = {
    "walk": {
        "fmd": walk_fmd,
        "mpjpe_nn": mpjpe_nn_walk,
        "mpjpe_vs_mean": mpjpe_mean_walk,
        "mean_pairwise_dist": var_walk["mean_pairwise_dist"],
        "joint_position_std": var_walk["joint_position_std"],
        "velocity_std": var_walk["velocity_std"],
        "n_real": int(len(real_walk)),
    },
    "jump": {
        "fmd": jump_fmd,
        "mpjpe_nn": mpjpe_nn_jump,
        "mpjpe_vs_mean": mpjpe_mean_jump,
        "mean_pairwise_dist": var_jump["mean_pairwise_dist"],
        "joint_position_std": var_jump["joint_position_std"],
        "velocity_std": var_jump["velocity_std"],
        "n_real": int(len(real_jump)),
    }
}

In [14]:
print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")

header = f"{'Metric':<35}"

for cls in results:
    header += f"  {cls.upper():>10}"

print(header)
print("-" * len(header))

metric_keys = [
    ("FMD", "fmd"),
    ("MPJPE (NN)", "mpjpe_nn"),
    ("MPJPE (vs mean real)", "mpjpe_vs_mean"),
    ("Diversity – pairwise", "mean_pairwise_dist"),
    ("Diversity – joint std", "joint_position_std"),
    ("Diversity – vel std", "velocity_std"),
]

for label, key in metric_keys:
    row = f"{label:<35}"

    for cls in results:
        row += f"  {results[cls][key]:>10.4f}"

    print(row)



SUMMARY
Metric                                     WALK        JUMP
-----------------------------------------------------------
FMD                                     35.0393    149.2054
MPJPE (NN)                               2.7084      2.1211
MPJPE (vs mean real)                    19.8097      6.1609
Diversity – pairwise                    52.0644     22.5965
Diversity – joint std                    9.5103      3.3395
Diversity – vel std                      0.9654      0.4784


In [15]:
print("\n\n" + "=" * 60)
print("MARKDOWN TABLE")
print("=" * 60)

md  = "| **Ruch** | **FMD** | **MPJPE** | **Var** |\n"
md += "|:--------:|--------:|----------:|--------:|\n"

for cls_name, res in results.items():
    md += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['joint_position_std']:.4f} |\n"
    )

print(md)



MARKDOWN TABLE
| **Ruch** | **FMD** | **MPJPE** | **Var** |
|:--------:|--------:|----------:|--------:|
| *walk* | 35.0393 | 2.7084 | 9.5103 |
| *jump* | 149.2054 | 2.1211 | 3.3395 |



In [16]:
print("-" * 60)
print("EXTENDED TABLE (all sub-metrics)\n")

ext  = "| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |\n"
ext += "|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|\n"

for cls_name, res in results.items():
    ext += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['mpjpe_vs_mean']:.4f} "
        f"| {res['mean_pairwise_dist']:.4f} "
        f"| {res['joint_position_std']:.4f} "
        f"| {res['velocity_std']:.4f} |\n"
    )

print(ext)

------------------------------------------------------------
EXTENDED TABLE (all sub-metrics)

| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |
|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|
| *walk* | 35.0393 | 2.7084 | 19.8097 | 52.0644 | 9.5103 | 0.9654 |
| *jump* | 149.2054 | 2.1211 | 6.1609 | 22.5965 | 3.3395 | 0.4784 |

